# Kaggle launcher: train on a free T4

This notebook only **launches** code from `src/` through the CLI. No model or training logic lives here
(the spec requires pure Python for the final code).

**One-time setup:** verify your phone number on Kaggle, then *Settings → Accelerator: GPU T4* and *Internet: On*.

**Long runs:** use *Save Version → Save & Run All (Commit)*. The run keeps going after you close the tab,
and everything in `/kaggle/working` appears in the version's *Output* tab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Change `BRANCH` if you want to run a feature branch.

In [ ]:
REPO = "https://github.com/karimmarwanw/From-Pixels-to-Language.git"
BRANCH = "main"
!git clone -q --branch $BRANCH $REPO /kaggle/working/project
%cd /kaggle/working/project
!git log --oneline -1

## 2. Install what Kaggle does not have
Kaggle already ships PyTorch with the right CUDA build, so we do **not** reinstall torch from
`requirements.txt` (that would download ~2 GB and can break the GPU setup). We add the rest.

In [ ]:
!pip install -q mlflow==3.17.2 datasets==5.1.0 imagehash pyspellchecker
import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

## 3. Settings for this session
The dataset cache goes to `/tmp` so the 1.1 GB download is not copied into the notebook output.

In [ ]:
CONFIG = "configs/cls_scratch.yaml"
COMMON = "data.cache_dir=/tmp/hf_cache data.num_workers=4"

## 4. Smoke test (about 2 minutes)
1 epoch on 256 images. Checks that data, model, MLflow and checkpoints all work before the long run.

In [ ]:
!python -m src.cli train --task classification --config $CONFIG --override $COMMON data.limit=256 train.epochs=1 run_name=smoke

## 5. Full training run
The scratch baseline: 40 epochs with early stopping on val accuracy.

In [ ]:
!python -m src.cli train --task classification --config $CONFIG --override $COMMON

## 6. Evaluate the best checkpoint on the validation set
(The test set is only used for the final chosen models.)

In [ ]:
import glob, os
run_dir = sorted(d for d in glob.glob("outputs/checkpoints/*") if not os.path.basename(d).startswith("smoke"))[-1]
print(run_dir)
!python -m src.cli evaluate --task classification --checkpoint {run_dir}/best.pt --split val

## 7. Package the results
Download `results.zip` from the *Output* tab. Upload `best.pt` to the team Google Drive. `mlflow.db` + `mlruns/` hold the MLflow runs.

In [ ]:
!zip -qr /kaggle/working/results.zip outputs/checkpoints outputs/predictions mlflow.db mlruns
!ls -lh /kaggle/working/results.zip

## If a session dies: resume
1. Open the previous version's *Output*, or upload your `results.zip` as a Kaggle Dataset, and add it as input.
2. Copy the run folder and `mlflow.db` back, then resume (same MLflow run, continues from the last finished epoch):

In [ ]:
# !cp -r /kaggle/input/<your-dataset>/outputs /kaggle/input/<your-dataset>/mlflow.db /kaggle/input/<your-dataset>/mlruns .
# !python -m src.cli train --task classification --config $CONFIG --override $COMMON --resume outputs/checkpoints/<run_dir>/last.pt